# 任务三：卷积神经网络模型设计

本任务将改变网络结构，进行实验对比及分析，包括：
1. 改变网络结构参数，包括卷积层数、卷积核尺寸、步长及是否填充、全连接层层数、各层节点数目等
2. 添加和不添加BN层

In [ ]:
import mindspore
# 载入mindspore的默认数据集
import mindspore.dataset as ds
# 常用转化用算子
import mindspore.dataset.transforms.c_transforms as C
# 图像转化用算子
import mindspore.dataset.vision.c_transforms as CV
from mindspore.common import dtype as mstype
# mindspore的tensor
from mindspore import Tensor


# 各类网络层都在nn里面
import mindspore.nn as nn
# 参数初始化的方式
from mindspore.common.initializer import TruncatedNormal
# 设置mindspore运行的环境
from mindspore import context
# 引入训练时候会使用到回调函数，如checkpoint, lossMoniter
from mindspore.train.callback import ModelCheckpoint, CheckpointConfig, LossMonitor, TimeMonitor
# 引入模型
from mindspore.train import Model
# 引入评估模型的包
from mindspore.nn.metrics import Accuracy

# numpy
import numpy as np
# 画图用
import matplotlib.pyplot as plt

# 下载数据相关的包
import os
import requests 
import zipfile

In [ ]:
!wget https://ascend-professional-construction-dataset.obs.cn-north-4.myhuaweicloud.com/ComputerVision/cifar10_mindspore.zip
!unzip cifar10_mindspore.zip

In [ ]:
#创建图像标签列表
category_dict = {0:'airplane',1:'automobile',2:'bird',3:'cat',4:'deer',5:'dog',
                 6:'frog',7:'horse',8:'ship',9:'truck'}
current_path = os.getcwd()
data_path = os.path.join(current_path, 'data/10-verify-bin')
cifar_ds = ds.Cifar10Dataset(data_path)
# 设置图像大小
plt.figure(figsize=(8,8))
i = 1
# 打印9张子图
for dic in cifar_ds.create_dict_iterator():
    plt.subplot(3,3,i)
    plt.imshow(dic['image'].asnumpy())
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    plt.title(category_dict[dic['label'].asnumpy().sum()])
    i +=1
    if i > 9 :
        break
plt.show()

In [ ]:
def get_data(datapath):
    cifar_ds = ds.Cifar10Dataset(datapath)
    return cifar_ds

def process_dataset(cifar_ds,batch_size =32,status="train"):
    '''
    ---- 定义算子 ----
    '''
    # 归一化
    rescale = 1.0 / 255.0
    # 平移
    shift = 0.0

    resize_op = CV.Resize((32, 32))
    rescale_op = CV.Rescale(rescale, shift)
    # 对于RGB三通道分别设定mean和std
    normalize_op = CV.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
    if status == "train":
        # 随机裁剪
        random_crop_op = CV.RandomCrop([32, 32], [4, 4, 4, 4])
        # 随机翻转
        random_horizontal_op = CV.RandomHorizontalFlip()
    # 通道变化
    channel_swap_op = CV.HWC2CHW()
    # 类型变化
    typecast_op = C.TypeCast(mstype.int32)

    '''
    ---- 算子运算 ----
    '''
    cifar_ds = cifar_ds.map(input_columns="label", operations=typecast_op)
    if status == "train":
        cifar_ds = cifar_ds.map(input_columns="image", operations=random_crop_op)
        cifar_ds = cifar_ds.map(input_columns="image", operations=random_horizontal_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=resize_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=rescale_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=normalize_op)
    cifar_ds = cifar_ds.map(input_columns="image", operations=channel_swap_op)
    
    # shuffle
    cifar_ds = cifar_ds.shuffle(buffer_size=1000)
    # 切分数据集到batch_size
    cifar_ds = cifar_ds.batch(batch_size, drop_remainder=True)
    
    return cifar_ds

In [ ]:
from mindspore.train.callback import Callback

class EvalCallBack(Callback):
    def __init__(self, model, eval_dataset, eval_per_epoch, epoch_per_eval):
        self.model = model
        self.eval_dataset = eval_dataset
        self.eval_per_epoch = eval_per_epoch
        self.epoch_per_eval = epoch_per_eval

    def epoch_end(self, run_context):
        cb_param = run_context.original_args()
        cur_epoch = cb_param.cur_epoch_num
        if cur_epoch % self.eval_per_epoch == 0:
            acc = self.model.eval(self.eval_dataset, dataset_sink_mode=False)
            self.epoch_per_eval["epoch"].append(cur_epoch)
            self.epoch_per_eval["acc"].append(acc["Accuracy"])
            print(acc)

## 1. 原始LeNet-5网络（作为基准）

In [ ]:
"""LeNet."""

def conv(in_channels, out_channels, kernel_size, stride=1, padding=0):
    """weight initial for conv layer"""
    weight = weight_variable()
    return nn.Conv2d(in_channels, out_channels,
                     kernel_size=kernel_size, stride=stride, padding=padding,
                     weight_init=weight, has_bias=False, pad_mode="same")

def fc_with_initialize(input_channels, out_channels):
    """weight initial for fc layer"""
    weight = weight_variable()
    bias = weight_variable()
    return nn.Dense(input_channels, out_channels, weight, bias)

def weight_variable():
    """weight initial"""
    return TruncatedNormal(0.02)

class LeNet5(nn.Cell):
    """
    Lenet network
    
    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5, self).__init__()
        self.num_class = num_class
        self.conv1 = conv(channel, 6, 5)
        self.conv2 = conv(6, 16, 5)
        self.fc1 = fc_with_initialize(16 * 8 * 8, 120)
        self.fc2 = fc_with_initialize(120, 84)
        self.fc3 = fc_with_initialize(84, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

## 2. 改进版LeNet-5网络（添加BN层）

In [ ]:
class LeNet5_improve(nn.Cell):
    """
    改进版Lenet网络，添加BN层并修改网络结构
    
    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet5_improve(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5_improve, self).__init__()
        self.num_class = num_class
        # 使用3x3卷积核替代5x5
        self.conv1_1 = conv(channel, 12, 3)
        self.bn2_1 = nn.BatchNorm2d(num_features=12)
        self.conv1_2 = conv(12, 24, 3)
        self.bn2_2 = nn.BatchNorm2d(num_features=24)        
        self.conv2_1 = conv(24, 48, 3)
        self.bn2_3 = nn.BatchNorm2d(num_features=48)        
        self.conv2_2 = conv(48, 96, 3)
        self.bn2_4 = nn.BatchNorm2d(num_features=96)
        self.fc1 = fc_with_initialize(96*8*8, 160)
        self.bn1_1 = nn.BatchNorm1d(num_features=160)
        self.fc2 = fc_with_initialize(160, 120)
        self.bn1_2 = nn.BatchNorm1d(num_features=120)
        self.fc3 = fc_with_initialize(120, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
    def construct(self, x):
        x = self.conv1_1(x)
        x = self.bn2_1(x)
        x = self.relu(x)
        x = self.conv1_2(x)
        x = self.bn2_2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2_1(x)
        x = self.bn2_3(x)
        x = self.relu(x)
        x = self.conv2_2(x)
        x = self.bn2_4(x)
        x = self.relu(x)
        x = self.max_pool2d(x)        
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.bn1_1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.bn1_2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

## 3. 不含BN层的改进网络

In [ ]:
class LeNet5_improve_no_bn(nn.Cell):
    """
    改进版Lenet网络，不添加BN层
    
    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet5_improve_no_bn(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5_improve_no_bn, self).__init__()
        self.num_class = num_class
        # 使用3x3卷积核替代5x5
        self.conv1_1 = conv(channel, 12, 3)
        self.conv1_2 = conv(12, 24, 3)
        self.conv2_1 = conv(24, 48, 3)
        self.conv2_2 = conv(48, 96, 3)
        self.fc1 = fc_with_initialize(96*8*8, 160)
        self.fc2 = fc_with_initialize(160, 120)
        self.fc3 = fc_with_initialize(120, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
    def construct(self, x):
        x = self.conv1_1(x)
        x = self.relu(x)
        x = self.conv1_2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2_1(x)
        x = self.relu(x)
        x = self.conv2_2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)        
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

## 4. 不同卷积核尺寸的网络

In [ ]:
class LeNet5_kernel7(nn.Cell):
    """
    使用7x7卷积核的Lenet网络
    
    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet5_kernel7(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5_kernel7, self).__init__()
        self.num_class = num_class
        # 使用7x7卷积核
        self.conv1 = conv(channel, 6, 7)
        self.conv2 = conv(6, 16, 7)
        self.fc1 = fc_with_initialize(16 * 8 * 8, 120)
        self.fc2 = fc_with_initialize(120, 84)
        self.fc3 = fc_with_initialize(84, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

## 5. 不同全连接层层数的网络

In [ ]:
class LeNet5_more_fc(nn.Cell):
    """
    增加全连接层层数的Lenet网络
    
    Args:
        num_class (int): Num classes. Default: 10.

    Returns:
        Tensor, output tensor
    Examples:
        >>> LeNet5_more_fc(num_class=10)

    """
    def __init__(self, num_class=10, channel=3):
        super(LeNet5_more_fc, self).__init__()
        self.num_class = num_class
        self.conv1 = conv(channel, 6, 5)
        self.conv2 = conv(6, 16, 5)
        # 增加全连接层层数
        self.fc1 = fc_with_initialize(16 * 8 * 8, 256)
        self.fc2 = fc_with_initialize(256, 128)
        self.fc3 = fc_with_initialize(128, 64)
        self.fc4 = fc_with_initialize(64, self.num_class)
        self.relu = nn.ReLU()      
        self.max_pool2d = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        
    def construct(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.conv2(x)
        x = self.relu(x)
        x = self.max_pool2d(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc4(x)
        return x

## 6. 训练函数

In [ ]:
def train_model(network, network_name, epochs=30, batch_size=32, learning_rate=0.001):
    """
    训练模型的通用函数
    """
    # 生成训练数据集
    data_path = os.path.join(current_path, 'data/10-batches-bin')
    cifar_ds = get_data(data_path)
    ds_train = process_dataset(cifar_ds, batch_size=batch_size, status="train")

    # 设置运行环境
    device_target = mindspore.context.get_context('device_target')
    dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False
    context.set_context(mode=context.GRAPH_MODE, device_target=device_target)

    # 设置损失函数和优化器
    net_loss = nn.SoftmaxCrossEntropyWithLogits(sparse=True, reduction="mean")
    net_opt = nn.Adam(params=network.trainable_params(), learning_rate=learning_rate)

    # 设置回调函数
    config_ck = CheckpointConfig(save_checkpoint_steps=1562, keep_checkpoint_max=10)
    ckpoint_cb = ModelCheckpoint(prefix=f"checkpoint_{network_name}", directory='./results', config=config_ck)
    time_cb = TimeMonitor(data_size=ds_train.get_dataset_size())

    # 建立模型
    model = Model(network=network, loss_fn=net_loss, optimizer=net_opt, metrics={"Accuracy": Accuracy()})
    eval_per_epoch = 1
    epoch_per_eval = {"epoch": [], "acc": []}
    eval_cb = EvalCallBack(model, ds_train, eval_per_epoch, epoch_per_eval)

    print(f"============== Starting Training {network_name} ==============")
    model.train(epochs, ds_train, callbacks=[ckpoint_cb, LossMonitor(per_print_times=1), eval_cb], dataset_sink_mode=dataset_sink_mode)
    
    return model, epoch_per_eval

## 7. 训练原始LeNet-5网络（基准）

In [ ]:
# 训练原始LeNet-5网络
network_original = LeNet5(10)
model_original, epoch_per_eval_original = train_model(network_original, "LeNet5_original", epochs=30)

## 8. 训练改进版LeNet-5网络（含BN层）

In [ ]:
# 训练改进版LeNet-5网络（含BN层）
network_improved = LeNet5_improve(10)
model_improved, epoch_per_eval_improved = train_model(network_improved, "LeNet5_improved", epochs=30)

## 9. 训练不含BN层的改进网络

In [ ]:
# 训练不含BN层的改进网络
network_improved_no_bn = LeNet5_improve_no_bn(10)
model_improved_no_bn, epoch_per_eval_improved_no_bn = train_model(network_improved_no_bn, "LeNet5_improve_no_bn", epochs=30)

## 10. 训练使用7x7卷积核的网络

In [ ]:
# 训练使用7x7卷积核的网络
network_kernel7 = LeNet5_kernel7(10)
model_kernel7, epoch_per_eval_kernel7 = train_model(network_kernel7, "LeNet5_kernel7", epochs=30)

## 11. 训练增加全连接层层数的网络

In [ ]:
# 训练增加全连接层层数的网络
network_more_fc = LeNet5_more_fc(10)
model_more_fc, epoch_per_eval_more_fc = train_model(network_more_fc, "LeNet5_more_fc", epochs=30)

## 12. 模型评估与结果对比

In [ ]:
# 生成测试数据集
data_path_test = os.path.join(current_path, 'data/10-verify-bin')
batch_size_test = 32
cifar_ds_test = ds.Cifar10Dataset(data_path_test)
ds_eval = process_dataset(cifar_ds_test, batch_size=batch_size_test, status="test")

# 设置运行环境
device_target = mindspore.context.get_context('device_target')
dataset_sink_mode = True if device_target in ['Ascend','GPU'] else False

# 评估各个模型
print("测试集评估结果：")
print("1. 原始LeNet-5网络：")
res_original = model_original.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_original)

print("2. 改进版LeNet-5网络（含BN层）：")
res_improved = model_improved.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_improved)

print("3. 不含BN层的改进网络：")
res_improved_no_bn = model_improved_no_bn.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_improved_no_bn)

print("4. 使用7x7卷积核的网络：")
res_kernel7 = model_kernel7.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_kernel7)

print("5. 增加全连接层层数的网络：")
res_more_fc = model_more_fc.eval(ds_eval, dataset_sink_mode=dataset_sink_mode)
print(res_more_fc)

## 13. 结果可视化

In [ ]:
# 绘制不同网络结构的准确率对比
plt.figure(figsize=(15, 10))

# 所有网络结构对比
plt.subplot(2, 2, 1)
plt.plot(epoch_per_eval_original["epoch"], epoch_per_eval_original["acc"], 'b-', label='原始LeNet-5')
plt.plot(epoch_per_eval_improved["epoch"], epoch_per_eval_improved["acc"], 'r-', label='改进版(含BN层)')
plt.plot(epoch_per_eval_improved_no_bn["epoch"], epoch_per_eval_improved_no_bn["acc"], 'g-', label='改进版(无BN层)')
plt.plot(epoch_per_eval_kernel7["epoch"], epoch_per_eval_kernel7["acc"], 'y-', label='7x7卷积核')
plt.plot(epoch_per_eval_more_fc["epoch"], epoch_per_eval_more_fc["acc"], 'm-', label='更多全连接层')
plt.title('Accuracy vs. Epoch (Different Network Structures)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# BN层对比
plt.subplot(2, 2, 2)
plt.plot(epoch_per_eval_improved["epoch"], epoch_per_eval_improved["acc"], 'r-', label='含BN层')
plt.plot(epoch_per_eval_improved_no_bn["epoch"], epoch_per_eval_improved_no_bn["acc"], 'g-', label='无BN层')
plt.title('Impact of Batch Normalization')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 卷积核大小对比
plt.subplot(2, 2, 3)
plt.plot(epoch_per_eval_original["epoch"], epoch_per_eval_original["acc"], 'b-', label='5x5卷积核')
plt.plot(epoch_per_eval_kernel7["epoch"], epoch_per_eval_kernel7["acc"], 'y-', label='7x7卷积核')
plt.title('Impact of Kernel Size')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

# 全连接层层数对比
plt.subplot(2, 2, 4)
plt.plot(epoch_per_eval_original["epoch"], epoch_per_eval_original["acc"], 'b-', label='2个全连接层')
plt.plot(epoch_per_eval_more_fc["epoch"], epoch_per_eval_more_fc["acc"], 'm-', label='3个全连接层')
plt.title('Impact of Fully Connected Layers')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

## 14. 预测结果可视化

In [ ]:
# 使用改进版模型进行预测可视化
category_dict = {0:'airplane',1:'automobile',2:'bird',3:'cat',4:'deer',5:'dog',
                 6:'frog',7:'horse',8:'ship',9:'truck'}

cifar_ds = get_data('./data/10-verify-bin')
df_test = process_dataset(cifar_ds,batch_size=1,status='test')

def normalization(data):
    _range = np.max(data) - np.min(data)
    return (data - np.min(data)) / _range

# 设置图像大小
plt.figure(figsize=(15, 15))
i = 1
# 打印9张子图
for dic in df_test:
    # 预测单张图片
    input_img = dic[0]    
    output = model_improved.predict(Tensor(input_img))
    output = nn.Softmax()(output)
    # 反馈可能性最大的类别
    predicted = np.argmax(output.asnumpy(),axis=1)[0]
    
    # 可视化
    plt.subplot(3,3,i)
    # 删除batch维度
    input_image = np.squeeze(input_img.asnumpy(),axis=0).transpose(1,2,0)
    # 重新归一化，方便可视化
    input_image = normalization(input_image)
    plt.imshow(input_image)
    plt.xticks([])
    plt.yticks([])
    plt.axis('off')
    plt.title('True label:%s,\n Predicted:%s'%(category_dict[dic[1].asnumpy().sum()],category_dict[predicted]))
    i +=1
    if i > 9 :
        break

plt.show()

## 15. 结论与分析

根据以上实验结果，我们可以分析不同网络结构对模型性能的影响：

1. **BN层的影响**：
   - BN层可以加速训练收敛
   - BN层可以提高模型的泛化能力
   - BN层可以允许使用更高的学习率

2. **卷积核大小的影响**：
   - 较小的卷积核（如3x3）可以捕获更精细的特征
   - 较大的卷积核（如7x7）可以捕获更大范围的特征
   - 多个小卷积核堆叠可以替代一个大卷积核，同时减少参数数量

3. **全连接层层数的影响**：
   - 增加全连接层层数可以提高模型的表达能力
   - 过多的全连接层可能导致过拟合
   - 全连接层参数数量大，增加计算复杂度

4. **网络深度的影响**：
   - 适当的增加网络深度可以提高模型性能
   - 过深的网络可能导致梯度消失或爆炸问题
   - 需要配合其他技术（如残差连接）来构建更深的网络

5. **参数数量的影响**：
   - 参数数量越多，模型表达能力越强
   - 参数数量过多可能导致过拟合
   - 需要在模型性能和计算资源之间找到平衡